# Module 1 — Data model and quality

Word guide Steps 1.1.1–1.1.7. Source conversions are in `scripts/get_data.py`; the model is `sql/01_model.sql`; the builder is `scripts/build_phase1.py`. This notebook reads their saved results so DBeaver can remain open. Raw inputs are preserved. Monetary values are USD.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
DATA = ROOT / 'exports/phase1'
def load(name):
    return pd.read_csv(DATA / f'{name}.csv')
manifest = json.loads((ROOT / 'docs/source_manifest.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'source': f['name'], 'rows': f['rows'], 'sha256': f['sha256']} for f in manifest['files']]))
print('Source commit:', manifest['commit'])

## 1. Learn the fields and profile the sources

IDs are labels. `sales_value` is retailer receipts; retailer-funded discounts combine loyalty discounts and coupon matching. Manufacturer coupons are reported separately. Inspect nulls before joining or replacing values.

In [ ]:
display(load('profile_transaction_summary').T)
columns = load('profile_columns')
display(columns[columns.null_rows > 0])
display(load('profile_relationships'))

## 2. Build the model and document cleaning

The model removes only quantity-zero/sales-zero voids. It retains free lines, zero-quantity lines with sales, unknown products and large quantities. Promotion conflicts use MULTIPLE and preserve recorded display/mailer exposure.

In [ ]:
display(load('fact_sales_sample').head(10))
display(load('profile_missing_products'))
display(load('profile_promotion_quality'))
display(Markdown((ROOT / 'docs/cleaning_log.md').read_text(encoding='utf-8')))

## 3. Review anomalies and time boundaries

Flags prompt inspection, not deletion. Local dates use America/New_York; UTC timestamps are retained. A missing purchase date does not prove a store was closed.

In [ ]:
display(load('daily_anomalies'))
display(load('basket_anomalies').sort_values('basket_value', ascending=False).head(10))
display(load('profile_missing_days'))
display(load('profile_campaign_windows'))

## 4. Reconcile the model

Every check must pass. In DBeaver connect to `grocery_phase1.duckdb` and inspect `sql/01d_inspect.sql`; use Ctrl+Enter for one query at a time. The builder reproduces the module in a fresh project copy.

In [ ]:
checks = load('phase1_checks')
assert checks.passed.astype(str).str.lower().eq('true').all()
print(f'{len(checks)} model checks passed')
display(checks)
display(Markdown((ROOT / 'docs/phase1_findings.md').read_text(encoding='utf-8')))